# Stage 2: align and fuse embedding views

This tutorial uses **synthetic data only**. Its scores illustrate the API, not biological findings.
We fit on training entities, transform unseen entities, then compare CCA, MAXVAR GCCA,
Procrustes and matched-width PCA with grouped nested validation. Real LINCS runs belong on the cluster.


In [1]:
import tempfile
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
from embpy.io.result import EmbeddingResult, EmbeddingProvenance
from embpy.tl.fusion import (CCAFusion, GCCAFusion, ProcrustesFusion, PCAFusion,
                            FusionConfig, subset_views)
from embpy.tl import benchmark_fusion, benchmark_fusion_retrieval

rng = np.random.default_rng(19)
n = 90
ids = tuple(f"synthetic_{i}" for i in range(n))
latent = rng.normal(size=(n, 3))
views = {
    name: EmbeddingResult(
        latent @ rng.normal(size=(3, width)) + 0.3 * rng.normal(size=(n, width)),
        ids, "sample", "synthetic_id", EmbeddingProvenance("synthetic:" + name))
    for name, width in [("a", 8), ("b", 6), ("c", 7)]
}
targets = pd.DataFrame(latent @ rng.normal(size=(3, 5)) + 0.2 * rng.normal(size=(n, 5)), index=ids)
labels = pd.Series(np.where(latent[:, 0] > 0, "positive", "negative"), index=ids)
groups = pd.Series([f"group_{i // 2}" for i in range(n)], index=ids)
{name: v.matrix.shape for name, v in views.items()}


{'a': (90, 8), 'b': (90, 6), 'c': (90, 7)}

## Train once; reuse the fitted maps

All views represent the same entities, even when their feature widths differ.
CCA takes exactly two views. GCCA and Procrustes accept two or more. Default L2 normalization
is followed by training-only centering. Every fitted view is required for new entities.


In [2]:
train = subset_views(views, ids[:60])
held_out = subset_views(views, ids[60:])
operators = {
    "cca": CCAFusion(3, regularization=0.1, pre_components=4),
    "gcca": GCCAFusion(3, regularization=0.1, pre_components=4),
    "procrustes": ProcrustesFusion(3, reference="a"),
}
outputs = {}
for name, operator in operators.items():
    keys = ("a", "b") if name == "cca" else tuple(views)
    operator.fit({k: train[k] for k in keys})
    outputs[name] = operator.transform({k: held_out[k] for k in reversed(keys)})
    assert outputs[name].entity_ids == ids[60:]
    assert outputs[name].matrix.shape == (30, 3)
{name: {"width": v.n_dims, "held_out_entities": v.n_entities} for name, v in outputs.items()}


{'cca': {'width': 3, 'held_out_entities': 30},
 'gcca': {'width': 3, 'held_out_entities': 30},
 'procrustes': {'width': 3, 'held_out_entities': 30}}

In [3]:
# Training diagnostics are not held-out task scores.
operators["cca"].canonical_correlations_, operators["gcca"].consensus_eigenvalues_


(array([0.94850945, 0.90901006, 0.6956475 ]),
 array([0.92367929, 0.86004711, 0.73341493]))

In [4]:
with tempfile.TemporaryDirectory() as directory:
    path = Path(directory) / "gcca.joblib"
    joblib.dump(operators["gcca"], path)
    restored = joblib.load(path)
    np.testing.assert_allclose(restored.transform(held_out).matrix, outputs["gcca"].matrix)
print("Saved and restored reusable maps successfully.")


Saved and restored reusable maps successfully.


## Paired benchmarks with nested parameter selection

Compare each method with PCA on the same views and width. The outer group split is shared
by every configuration. Fusion and Ridge hyperparameters are selected only within outer
training folds. For retrieval, inner labels select fusion parameters by precision@k.
The fusion operators themselves never use labels. The synthetic grouping here merely
illustrates the API; use biological compound/donor groups for real data.


In [5]:
configs = {
    "pair_pca": FusionConfig(PCAFusion(3, normalization="l2"), ("a", "b")),
    "pair_cca": FusionConfig(CCAFusion(3, pre_components=4), ("a", "b"),
                             {"regularization": [0.01, 0.1]}),
    "multi_pca": PCAFusion(3, normalization="l2"),
    "multi_gcca": FusionConfig(GCCAFusion(3, pre_components=4),
                               param_grid={"regularization": [0.01, 0.1]}),
    "multi_procrustes": ProcrustesFusion(3, reference="a"),
}
with tempfile.TemporaryDirectory() as cache:
    regression = benchmark_fusion(
        views, targets, groups=groups, fusions=configs, cv=3, seeds=(0,),
        single_n_components=3, tune=True, param_grid={"alpha": [0.1, 1]}, cache_dir=cache)
regression.scores.groupby("configuration")[["mse", "pearson", "output_dim"]].mean()


,mse,pearson,output_dim
configuration,,,
multi_gcca,0.471228,0.972449,3.0
multi_pca,0.458527,0.973599,3.0
multi_procrustes,0.455687,0.973258,3.0
pair_cca,0.509228,0.965331,3.0
pair_pca,0.491033,0.967667,3.0
single:a,0.579196,0.953927,3.0
single:b,0.714928,0.908397,3.0
single:c,0.529115,0.961837,3.0


In [6]:
retrieval = benchmark_fusion_retrieval(
    views, labels, groups=groups, fusions=configs, cv=3, seeds=(0,),
    k=5, tune=True, single_n_components=3)
assert retrieval.splits.equals(regression.splits)
for _, split in regression.splits.groupby(["seed", "fold"]):
    assert set(split.loc[split.partition == "train", "group"]).isdisjoint(
        split.loc[split.partition == "test", "group"])
retrieval.scores.groupby("configuration")[["precision_at_k", "hit_rate_at_k"]].mean()


,precision_at_k,hit_rate_at_k
configuration,,
multi_gcca,0.871111,1.000000
multi_pca,0.875556,1.000000
multi_procrustes,0.866667,1.000000
pair_cca,0.873333,1.000000
pair_pca,0.895556,1.000000
rank_mean,0.877778,1.000000
score_mean,0.877778,1.000000
single:a,0.804444,0.988889
single:b,0.937778,0.988889


## Inspect the record and interpret it carefully

`best_params` records inner-fold selections; `splits` records actual entity and group assignments.
`projection_coefficients` counts stored projection-map entries, not independent degrees of freedom.
No ranking in this small synthetic example establishes a biological advantage. For real data,
retain source hashes, cohort exclusions, repeated grouped splits, and matched-view controls.
See `docs/fusion_alignment_benchmark_lincs.md` for the cluster benchmark.


In [7]:
regression.scores[["configuration", "fold", "best_params", "projection_coefficients"]].head(8)


,configuration,fold,best_params,projection_coefficients
0,single:a,0,{'alpha': 0.1},24
1,single:b,0,{'alpha': 0.1},18
2,single:c,0,{'alpha': 0.1},21
3,pair_pca,0,{'alpha': 0.1},42
4,pair_cca,0,"{'alpha': 0.1, 'fusion__regularization': 0.1}",42
5,multi_pca,0,{'alpha': 0.1},63
6,multi_gcca,0,"{'alpha': 0.1, 'fusion__regularization': 0.1}",63
7,multi_procrustes,0,{'alpha': 0.1},63
